# Reproducing Lesica et al. (2007) from dandiset 002028

**Paper.** Lesica NA, Jin J, Weng C, Yeh C-I, Butts DA, Stanley GB, Alonso J-M (2007). Adaptation to
stimulus contrast and correlations during natural visual stimulation. *Neuron* 55:479-491.
[doi:10.1016/j.neuron.2007.07.013](https://doi.org/10.1016/j.neuron.2007.07.013). The claims file
(`claims/lesica-2007.yaml`) was written from the version of record and its Supplemental Data.

**Dandiset.** [DANDI:002028](https://dandiarchive.org/dandiset/002028/0.261102.1631), version
0.261102.1631. The paper cites no version (it predates the dandiset, created in 2026). This is the
only published version and is pinned; the draft has the same 33 assets and size.

**What the dandiset holds.** Cat LGN single units from four animals (Expt1 to Expt4), with a
stimulus library of the movie and noise frames. The paper's data are in Expt1 and Expt2 (see the
data check): the 15 min movie at two contrasts, the long binary noise and the 12 s repeats (Expt1),
and the 90 s repeats (Expt2). Expt3 and Expt4 belong to the dandiset's other papers.

**Method in brief.** RFs (21 x 21 pixels, 12 lags of one frame) are estimated as the mean of nine
segment estimates, by the batch ridge solution that the paper's recursive least squares converges
to, with the same number of spikes at high and low contrast; NLs from the filtered stimulus in 8 ms
bins, groups of 250; a half-wave rectifier fit gives gain and offset. Reliability and sparseness
come from the repeats. Every parameter is cited in `lesica.py`; the choices the paper leaves open
are listed in the attempt record as deviations.

| Claim | Tested here |
|---|---|
| sample-composition | tested |
| firing-rate-contrast, firing-rate-lc-ns-wn-similar | tested |
| ln-prediction-accuracy | tested |
| fig3b, fig3c, fig3d (six claims) | tested |
| surround-temporal-correlation, figs1-property-correlations | tested |
| fig4b, fig4c, fig4d (six claims) | tested |
| fig5-within-trial-contrast | blocked (not run within the author's time) |
| fig6a, fig6b (four claims) | tested |
| fig7c-effective-contrast, fig7d-reliability-tracks-effective-contrast | tested |
| fig8a-filtered-kurtosis, fig8-model-sparseness-shuffle, fig8b, fig8de | tested |
| disc-lcns-hcwn-reliability-sparseness, disc-lcns-hcwn-rf-nl-differences | tested |
| methods-latency-biphasic, methods-offset-varies-across-animals | tested |
| celltype claims (four) | tested |
| methods-spherical-asymmetry | blocked (not run within the author's time) |

In [ ]:
import json
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import lesica as L

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()
assets = {a["path"]: a for a in L.assets()}
print(f"dandiset {L.DANDISET} version {L.VERSION}: {len(assets)} assets")
lib = L.open_asset(assets["sub-Expt1/sub-Expt1_ses-StimulusLibrary_image.nwb"]["asset_id"])
T = L.load_templates(lib, ("NAT", "WN"))
clips = {k: lib["stimulus/templates"][k]["data"][()] for k in ["clip_nr_b1", "clip_nr_b2", "clip_nr_b3",
                                                               "clip_wr_b1", "clip_wr_b2", "clip_wr_b3"]}
lib_rates = {k: float(lib["stimulus/templates"][k]["starting_time"].attrs["rate"]) for k in ["NAT", "WN"]}
lib.close()
print(f"stimulus library read in {time.time() - started:.0f} s")

## Data check

Every recording file is opened and its trials table, units and stimulus series are read (the
files' full contents were listed while preparing; the columns used are named in `lesica.py`).
Each check of the metadata against the values is stated under its output.

In [ ]:
files = sorted(p for p in assets if "ecephys" in p)
inv = []
for p in files:
    f = L.open_asset(assets[p]["asset_id"])
    s = L.load_session(f)
    tr = s["trials"]
    for i in range(len(tr["trial_key"])):
        if tr["modality"][i] in ("NAT", "SBN") and not tr["derived"][i]:
            fr = tr["frame_times"][i]
            inv.append({"file": p.split("/")[1].replace("_ecephys.nwb", ""), "subject": s["subject"],
                        "units": len(s["units"]["unit_name"]), "code": tr["code"][i], "regime": tr["regime"][i],
                        "contrast": tr["contrast"][i], "rms": tr["contrast_rms"][i],
                        "template": tr["stim_template"][i], "frames": f'{tr["stim_start"][i]}-{tr["stim_stop"][i]}',
                        "seconds": round(float(tr["duration_s"][i])), "n_reps": tr["n_reps"][i],
                        "frame_hz": round(1 / np.median(np.diff(fr[0])), 3) if fr is not None else np.nan,
                        "frames_in_order": bool(np.all(np.diff(fr[1]) == 1)) if fr is not None else None})
    f.close()
inv = pd.DataFrame(inv)
print("Units per file and subject:")
print(inv.groupby(["subject", "file"]).units.first().to_string())
print("\nNatural movie (NAT) and binary noise (SBN) recordings, by subject:")
print(inv.groupby(["subject", "regime", "template", "contrast", "rms", "seconds", "n_reps"], dropna=False).size().to_string())

Expt1 holds, in penetrations a11, a13 and a14, the 15 min movie at RMS contrast 0.40 and
0.15, the long binary noise at 0.55 and 0.20, and 120 repetitions of three 250-frame clips of
each stimulus at each contrast (12.5 s in all: the paper's "120 repeated trials of different
12 s segments"). Penetrations b11, b13 and b14 hold the 15 min movie (b11 also the noise, b14
only its high contrast), without repeats. Expt2 holds, in all eight penetrations, the 15 min movie
at both contrasts and 24 repetitions of a 90 s movie segment at both contrasts (the paper's 26-cell
cross-validation set). Expt3 and Expt4 hold no 15 min movie at two contrasts: they are the data of
the other three papers the dandiset supports. So the paper's data are in Expt1 and Expt2, two
animals. The contrasts match the paper's 0.40, 0.15, 0.55 and 0.20.

Differences from the paper's description, from the table: the long noise lasts 600 s (high
contrast) and 900 s (low contrast), where the paper says 6 min; the first 6 min are used. Frames
are shown at 59.96 Hz (the paper: 120 Hz refresh, "pixel intensities updated every other
refresh", which is 60 Hz; its "50 Hz" is the movie's interpolated rate).

In [ ]:
nat = T["NAT"].reshape(len(T["NAT"]), -1)
fs = nat.std(1)
print("NAT template:", T["NAT"].shape, "frame SD percentiles 0/1/50/99/100:", np.round(np.percentile(fs, [0, 1, 50, 99, 100]), 3),
      "; frame means range", np.round([nat.mean(1).min(), nat.mean(1).max()], 3), "; frames with SD < 0.5:", int((fs < 0.5).sum()))
print("WN template values:", np.unique(T["WN"][:200]), "; library frame rates:", lib_rates)
same = [k for k in clips if any(np.allclose(clips[k][0], T["NAT"][j]) for j in range(0, len(T["NAT"]), 1))] if False else []
print("Clip shapes:", {k: v.shape for k, v in clips.items()})
print("\nRMS contrast of a frame is SD/mean; with a mean of 125 gray levels, a movie frame at 0.40 has SD 50 gray levels.")
print("Stimuli are used as 125 x rms x template (gray levels about the mean).")

The movie template holds frames of zero mean and unit SD (the library says "mean-subtracted";
each frame is rescaled, as the paper says "each frame was rescaled to have an RMS contrast"). A
few frames are blank, listed above. The noise template is +-1.

In [ ]:
# Clock: the 90 s repeats of Expt2 are stated to show movie frames 0-5400, the first 90 s of the
# 15 min movie. If that and the clocks hold, the repeat PSTH (one bin per frame) correlates with
# the response to the first 5400 frames of the unique movie at zero shift and not at others.
# Computed in the main pass below (clock_cc, shifts -6..6 frames); shown after it.
print("see the clock check after the main pass")

## Choice of the ridge penalty

The paper estimates RFs by recursive least squares with initial conditions g = 0 and K = D x I.
Its final estimate equals the batch ridge solution with penalty 1/D, which is computed here
directly (faster, same result). The paper's D = 0.001 refers to its own stimulus scale and the
"+1" noise term of its gain equation, which this attempt cannot reconstruct, and with the penalty
of 1000 in gray levels the 5292-parameter estimate from one ninth of the data is not determined
(fewer frames than parameters). The paper chose D as "the value of D that produced the RF
estimates that provided the most accurate predictions of responses to natural scene movies"
(Estimation of Receptive Fields, paragraph 3); the same rule is applied here, on held-out movie
data (the last 10% of the 15 min movie), so as not to use the 90 s test segments. The penalty is
set relative to the stimulus variance, the same in every condition (an assumption: a fixed penalty
in gray levels would regularize the low-contrast RFs about seven times more strongly than the
high-contrast ones and bias the comparisons between contrasts).

In [ ]:
cases, labels = [], []
for p, units in [("sub-Expt1/sub-Expt1_ses-a13_ecephys.nwb", [0, 1]), ("sub-Expt2/sub-Expt2_ses-a11_ecephys.nwb", [1, 4])]:
    f = L.open_asset(assets[p]["asset_id"])
    s = L.load_session(f)
    tr = s["trials"]
    for c in ("H", "L"):
        i = L._trial(tr, "ns", c)
        ft, idx = tr["frame_times"][i]
        F = L._frames(T["NAT"], idx)
        for u in units:
            r = L.frame_counts(s["units"]["spikes"][u], ft)
            r0, c0 = L.window(L.locate(L._frames(T["NAT"], tr["frame_times"][L._trial(tr, "ns", "H")][1]),
                                       L.frame_counts(s["units"]["spikes"][u], tr["frame_times"][L._trial(tr, "ns", "H")][0])))
            S = np.ascontiguousarray(F[:, r0:r0 + L.WIN, c0:c0 + L.WIN]).reshape(len(F), -1) * np.float32(125 * float(tr["contrast_rms"][i]))
            cases.append((S, r))
            labels.append(f"{s['subject']} {s['session']} {s['units']['unit_name'][u]} {c}")
    f.close()
lams = [1000.0, 3000.0, 10000.0, 30000.0]
t0 = time.time()
cc = L.choose_ridge(cases, lams)
del cases
tab = pd.DataFrame(cc, index=labels, columns=[f"{x:g}" for x in lams])
print(tab.round(3).to_string())
print("mean:", tab.mean().round(4).to_dict())
L.RIDGE_REL = lams[int(np.argmax(cc.mean(0)))]
print(f"chosen penalty: {L.RIDGE_REL:g} x stimulus variance ({time.time() - t0:.0f} s)")

## Main pass

For every unit of the eleven penetrations with the 15 min movie at both contrasts (Expt1 a11, a13,
a14, b11, b13, b14; Expt2 a11, a12, b11, c11, d11, e21, f31, g31): spike counts per frame, the RF
window (21 x 21 pixels centred on the peak of the spike-triggered average of the high-contrast
movie), RFs from the high- and low-contrast movie with the same number of spikes, each the mean of
nine segment estimates, the RF from shuffled responses under the five seeds of
`sample-composition`, and the NL; for the Expt1 penetrations with repeats (a11, a13, a14) the same
for the first 6 min of the noise; and the repeat responses. In Expt2 the first 5400 movie frames,
which the 90 s repeats show again, are left out of the RF and NL estimates. Penetrations run in
parallel, one BLAS thread each.

In [ ]:
seeds = [L.seed("sample-composition") + k for k in (0, 1000, 2000, 3000, 4000)]
jobs = []
for e, sess in L.SESSIONS.items():
    for j, p in enumerate(sess):
        path = f"sub-{e}/sub-{e}_ses-{p}_ecephys.nwb"
        kw = dict(ns_skip=5400 if e == "Expt2" else 0, wn=(e == "Expt1" and p in ("a11", "a13", "a14")), key=(int(e[-1]), j))
        jobs.append((path, assets[path]["asset_id"], seeds, kw))
jobs.sort(key=lambda j: -(2 * j[3]["wn"] + 1))
t0 = time.time()
results = L.run_all(T, jobs)
print(f"main pass: {time.time() - t0:.0f} s")

In [ ]:
units = {(d["subject"], d["session"], d["unit_name"]): d for _, _, out in results for d in out}
df = pd.DataFrame(L.unit_table(results))
print("units:", len(df), "; units with an error:", int(df.error.notna().sum()), df.error.dropna().tolist())
for c in L.CONDITIONS:
    n = df[c + "_snr"].notna().sum() if c + "_snr" in df else 0
    print(f"{c}: RFs estimated for {n} units; seconds used {df[c + '_seconds'].min():.0f}-{df[c + '_seconds'].max():.0f}; "
          f"spikes {df[c + '_n_spikes'].min():.0f}-{df[c + '_n_spikes'].max():.0f}")
print("\nHC and LC spike counts used are equal by construction; largest difference:",
      float((df.ns_H_n_spikes - df.ns_L_n_spikes).abs().max()), "spikes")
cl = np.array([units[k]["clock_cc"] for k in units if "clock_cc" in units[k]])
print("\nClock check, Expt2 (90 s repeats against movie frames 0-5400), median correlation over", len(cl), "units by shift (frames):")
print(dict(zip(range(-6, 7), np.round(np.nanmedian(cl, 0), 3))))
print("units whose best shift is 0:", int((np.nanargmax(cl, 1) == 6).sum()), "of", len(cl))

The clock check shows the 90 s repeats are the first 90 s of the 15 min movie (correlation
peaks at zero shift); the paper calls them "a different 90 s movie segment". For Expt1, which has
no shared segment, the RFs themselves are the check: a peak at 2 to 3 frames (33 to 50 ms) after
the stimulus with a clear centre, shown below, would not arise with a broken clock.

In [ ]:
lat = df[["ns_H_latency", "ns_L_latency", "wn_H_latency", "wn_L_latency"]] * 1000
print("RF peak latency (ms), by subject:")
print(df.assign(**{c: lat[c] for c in lat}).groupby("subject")[list(lat)].describe().round(1).T.to_string())

## sample-composition

> The 15 min movie analyses include 68 LGN cells whose RF peak is at least ten times the noise, 44
> ON-center (19 X, 19 Y, 6 unclassified) and 24 OFF-center (11 X, 8 Y, 5 unclassified); 26 of them
> have the 90 s repeats and 31 have white noise and the 12 s repeats.

Inclusion: the largest absolute value of the HC and of the LC movie RF over the SD of the RF from
shuffled responses, at least 10; readings (a) both contrasts, (b) either. ON or OFF is the sign of
the maximal point of the HC movie RF (the files carry no ON/OFF label); X or Y is the units table's
`xy_class`. The downstream claims use reading (a).

In [ ]:
inc_a = (df.ns_H_snr >= 10) & (df.ns_L_snr >= 10)
inc_b = (df.ns_H_snr >= 10) | (df.ns_L_snr >= 10)
print(f"Units with both 15 min movies: {len(df)}; passing at both contrasts (reading a): {inc_a.sum()}; at either (b): {inc_b.sum()}")
for k, sd in enumerate((0, 1000, 2000, 3000, 4000)):
    a = df.ns_H_snrs.apply(lambda x: x[k]) >= 10
    b = df.ns_L_snrs.apply(lambda x: x[k]) >= 10
    print(f"  seed + {sd}: both {int((a & b).sum())}, either {int((a | b).sum())}")
print("peak/noise percentiles, HC:", np.round(np.percentile(df.ns_H_snr, [5, 25, 50, 75, 95]), 1),
      " LC:", np.round(np.percentile(df.ns_L_snr, [5, 25, 50, 75, 95]), 1))
for thr in (10, 15, 20, 25):
    print(f"  threshold {thr}: both {int(((df.ns_H_snr >= thr) & (df.ns_L_snr >= thr)).sum())}")
df["onoff"] = np.where(df.ns_H_sign > 0, "ON", "OFF")
df["xy"] = df.xy_class.replace("", "unclassified").str.upper().replace("UNCLASSIFIED", "unclassified")
print("units whose HC and LC movie RFs have opposite signs at their maximal point:", int((df.ns_H_sign != df.ns_L_sign).sum()))
for name, m in [("reading a (both)", inc_a), ("reading b (either)", inc_b)]:
    d = df[m]
    print(f"\n{name}: {len(d)} cells; by subject {d.subject.value_counts().to_dict()}")
    print(pd.crosstab(d.onoff, d.xy, margins=True).to_string())
    print("with the long noise and the 12 s repeats (Expt1 a11, a13, a14):", int(d.session.isin(["a11", "a13", "a14"]).where(d.subject == "Expt1", False).sum()),
          "; with the 90 s repeats (Expt2):", int((d.subject == "Expt2").sum()))
inc = df[inc_a].copy()
wn = inc[(inc.subject == "Expt1") & inc.session.isin(["a11", "a13", "a14"])].copy()
ex2 = inc[inc.subject == "Expt2"].copy()

In [ ]:
# Units recorded in more than one penetration: the same electrode in a11, a13 and a14 of Expt1
# ("electrode adjusted to isolate units better, same presumed eccentricity"), and Expt2 a11 and
# a12. Units on the same electrode are compared by the correlation of their HC movie RFs placed
# in screen coordinates.
def screen_rf(d):
    rf = np.zeros((L.NLAGS, 48, 48))
    r0, c0 = d["window"]
    rf[:, r0:r0 + L.WIN, c0:c0 + L.WIN] = d["ns_H"]["rf"].reshape(L.NLAGS, L.WIN, L.WIN)
    return rf.ravel()
pairs = []
keys = [k for k in units if "ns_H" in units[k] and "rf" in units[k]["ns_H"]]
for i, a in enumerate(keys):
    for b in keys[i + 1:]:
        if a[0] == b[0] and a[1] != b[1] and units[a]["electrode_number"] == units[b]["electrode_number"]:
            pairs.append({"a": " ".join(a), "b": " ".join(b), "r": np.corrcoef(screen_rf(units[a]), screen_rf(units[b]))[0, 1]})
pairs = pd.DataFrame(pairs)
print("pairs on the same electrode in different penetrations:", len(pairs), "; RF correlation > 0.8:", int((pairs.r > 0.8).sum()))
print(pairs[pairs.r > 0.8].round(3).to_string(index=False))
print("possible_duplicate_of entries in the units tables:", df.possible_duplicate_of.replace("", np.nan).dropna().tolist())

**Result: not reproduced.** 111 of the 143 units pass the inclusion rule at both contrasts (133 at
either), against the paper's 68, under all five seeds of the shuffles (111 each). The subset with the
long noise and the 12 s repeats is 31 cells, as in the paper; the subset with the 90 s repeats is 58,
against 26. Among the 111, 55 are ON and 56 OFF (paper 44 and 24), and 60 X, 43 Y and 8 unclassified
(paper 30, 27 and 11). The count depends strongly on the noise estimate and the regularization (61
cells pass at a threshold of 15). No pair of units on the same electrode in different penetrations has
RFs that correlate above 0.8, so no unit appears to have been counted twice.

## firing-rate-contrast and firing-rate-lc-ns-wn-similar

> Across the 31 cells with all four stimuli, mean firing rates are higher at high than at low
> contrast for both natural movies (10.5 against 6.5 Hz) and white noise (8.2 against 5.7 Hz).

> Mean firing rates to low-contrast natural movies and low-contrast white noise do not differ across
> the 31 cells.

The 31 cells are the included cells of Expt1 a11, a13 and a14, the only penetrations with the long
noise and the 12 s repeats. Readings P2: (a) the long segments (15 min movie, first 6 min of noise),
(b) the 12 s repeats.

In [ ]:
rep = {k: L.repeat_measures(units[(r.subject, r.session, r.unit_name)]) for k, r in wn.iterrows()}
for c in L.CONDITIONS:
    wn["rep_rate_" + c] = [rep[k][c]["rate"] for k in wn.index]
    wn["rel_a_" + c] = [rep[k][c]["rel_a"] for k in wn.index]
    wn["rel_b_" + c] = [rep[k][c]["rel_b"] for k in wn.index]
    wn["sparse_" + c] = [rep[k][c]["sparse"] for k in wn.index]
print("valid repetitions per block, range over cells:", min(min(rep[k][c]["n_reps"]) for k in wn.index for c in L.CONDITIONS),
      "-", max(max(rep[k][c]["n_reps"]) for k in wn.index for c in L.CONDITIONS))
def paired(a, b):
    t = stats.ttest_rel(a, b)
    return t.statistic, t.pvalue
rate_rows = []
for reading, cols in [("a: long segments", ["rate_ns_H", "rate_ns_L", "rate_wn_H", "rate_wn_L"]),
                      ("b: 12 s repeats", ["rep_rate_ns_H", "rep_rate_ns_L", "rep_rate_wn_H", "rep_rate_wn_L"])]:
    m = wn[cols].mean(); s = wn[cols].std()
    tns, pns = paired(wn[cols[0]], wn[cols[1]]); tw, pw = paired(wn[cols[2]], wn[cols[3]]); tc, pc = paired(wn[cols[1]], wn[cols[3]])
    rate_rows.append({"reading": reading, "n": len(wn), "HC NS": f"{m.iloc[0]:.2f} ({s.iloc[0]:.2f})", "LC NS": f"{m.iloc[1]:.2f} ({s.iloc[1]:.2f})",
                      "HC WN": f"{m.iloc[2]:.2f} ({s.iloc[2]:.2f})", "LC WN": f"{m.iloc[3]:.2f} ({s.iloc[3]:.2f})",
                      "p NS HC>LC": pns, "t NS": tns, "p WN HC>LC": pw, "t WN": tw, "p LC NS vs LC WN": pc, "LC NS - LC WN": m.iloc[1] - m.iloc[3]})
rates = pd.DataFrame(rate_rows)
print(rates.to_string(index=False))
print("\nby penetration (reading a):")
print(wn.groupby("session")[["rate_ns_H", "rate_ns_L", "rate_wn_H", "rate_wn_L"]].mean().round(2).to_string())
print("\n15 min movie rates over all", len(inc), "included cells, by subject:")
print(inc.groupby("subject")[["rate_ns_H", "rate_ns_L"]].agg(["mean", "std", "count"]).round(2).to_string())

**Result: partially reproduced.** Over the 31 cells and the long segments the four means are 8.68,
6.11, 6.54 and 5.35 Hz, all within the paper's SDs of 10.5, 6.5, 8.2 and 5.7 Hz. HC is higher than LC
for the movie (p = 5.0e-07) and for the noise (p = 0.0022), so the noise test misses the paper's
p < 0.001. Over the 12 s repeats the movie test gives p = 0.022.

In [ ]:
for _, r in rates.iterrows():
    print(f"{r.reading}: LC NS against LC WN, difference {r['LC NS - LC WN']:.2f} Hz, paired t p = {r['p LC NS vs LC WN']:.3g}; "
          f"p > 0.05: {r['p LC NS vs LC WN'] > 0.05}; p > 0.2: {r['p LC NS vs LC WN'] > 0.2}")

**Result: reproduced, under the long-segment reading (an assumption).** With rates over the long
segments the LC NS and LC WN rates differ by 0.75 Hz, paired t p = 0.063 (above 0.05, not above the
paper's 0.2). With rates over the 12 s repeats they differ by 1.37 Hz, p = 0.0013. The readings
disagree; the long segments are adopted, as the paper's comparison concerns the stimuli used to fit
the RFs and NLs.

## ln-prediction-accuracy

> LN models fitted from the 15 min movie predict responses to a novel 90 s movie segment with
> correlation coefficients of 0.70 at high and 0.76 at low contrast (26 cells, 8 ms bins).

Included cells of Expt2. The RF and the mean NL of each contrast (from movie frames 5400 to 54000,
so that the test frames 0-5400 are not in the training data) predict the 8 ms rate during the 24
repeats. Readings: (a) correlation with the trial-averaged response, (b) mean over trials of the
correlation with each trial.

In [ ]:
def predict(d, c):
    """LN prediction of the 90 s repeats: the unit's RF and mean NL (nine segments) for contrast c,
    applied to movie frames 0-5400 as shown in the repeats; 8 ms bins."""
    r0, c0 = d["window"]
    a, b = d["rep90_frames"]
    F = T["NAT"][a:b, r0:r0 + L.WIN, c0:c0 + L.WIN].reshape(b - a, -1) * np.float32(125 * d["rms"]["ns_" + c])
    y = L.filtered(F, d["ns_" + c]["rf"])
    R = d["rep90_" + c].astype(float)
    tc = (np.arange(R.shape[1]) + 0.5) * L.BIN
    fr = np.floor(tc / L.REP_DT).astype(int)
    ok = (fr >= L.NLAGS - 1) & (fr < len(y))
    pred = np.interp(y[fr[ok]], d["ns_" + c]["nl_x"], d["ns_" + c]["nl_y"])
    Rk = R[:, ok]
    ca = np.corrcoef(pred, Rk.mean(0))[0, 1]
    cb = np.nanmean([np.corrcoef(pred, x)[0, 1] for x in Rk if x.std() > 0])
    return ca, cb, len(R)
pr = []
for k, r in ex2.iterrows():
    d = units[(r.subject, r.session, r.unit_name)]
    row = {"session": r.session, "unit": r.unit_name, "xy": r.xy}
    for c in ("H", "L"):
        row[f"a_{c}"], row[f"b_{c}"], row[f"reps_{c}"] = predict(d, c)
    pr.append(row)
pr = pd.DataFrame(pr)
print(f"{len(pr)} cells; repetitions per contrast {pr.reps_H.min()}-{pr.reps_H.max()}")
for rd in ("a", "b"):
    print(f"reading {rd}: HC {pr[f'{rd}_H'].mean():.3f} (SD {pr[f'{rd}_H'].std():.3f}), LC {pr[f'{rd}_L'].mean():.3f} (SD {pr[f'{rd}_L'].std():.3f})")
print("\nby penetration (reading a):")
print(pr.groupby("session")[["a_H", "a_L"]].agg(["mean", "count"]).round(3).to_string())

**Result: not reproduced.** Over the 58 included Expt2 cells, the LN predictions correlate with the
trial-averaged 8 ms response at 0.477 (HC) and 0.532 (LC), below the paper's 0.70 and 0.76 (tolerance
0.07); with single trials, 0.282 and 0.259. LC is higher than HC under reading (a), as in the paper.

## Receptive field properties (Figure 3)

> fig3b-spatial-width-contrast: A decrease in contrast has no significant effect on the width of the
> spatial RF center, for natural movies (68 cells) or white noise (31 cells).

> fig3b-spatial-width-correlations: A decrease in correlations (low-contrast natural movie to
> low-contrast white noise) decreases the width of the spatial RF center by 20% on average (31 cells).

> fig3c-surround-contrast: A decrease in contrast decreases the relative strength of the RF surround
> by 17% for natural movies and 18% for white noise.

> fig3c-surround-correlations: A decrease in correlations decreases the relative strength of the RF
> surround by 34%.

> fig3d-temporal-width-contrast: A decrease in contrast widens the temporal profile of the RF center
> (slower dynamics) by 16% for natural movies and 15% for white noise.

> fig3d-temporal-width-correlations: A decrease in correlations widens the temporal profile of the RF
> center by 15%.

Each table gives, per comparison, the means, `pct_a` (P1 a, change of the means), `pct_b` (P1 b,
mean of per-cell changes), `ratio_c` (ratio of the means, %), and the paired t test. The NS HC to
LC comparison is shown over all included cells (the claim's 68) and over the 31 Expt1 cells.
Surround/center ratio readings: `_a` the surround's largest absolute value, `_b` its extreme of the
sign opposite to the centre.

In [ ]:
def compare(prop, a, b, data):
    x, y = data[a + "_" + prop].astype(float), data[b + "_" + prop].astype(float)
    group = f"all {len(inc)}" if len(data) == len(inc) else f"{len(data)} (Expt1)"
    ok = x.notna() & y.notna()
    t = stats.ttest_rel(x[ok], y[ok])
    pc = L.pct_change(x[ok], y[ok])
    return {"prop": prop, "comparison": f"{a}->{b}", "n": int(ok.sum()), "mean_a": x[ok].mean(), "mean_b": y[ok].mean(),
            "sd_a": x[ok].std(), "sd_b": y[ok].std(), "pct_a": pc["a"], "pct_b": pc["b"], "ratio_c": pc["c"], "t": t.statistic, "p": t.pvalue, "group": group}
rows = []
for prop in L.PROPS:
    rows.append(compare(prop, "ns_H", "ns_L", inc))
    rows.append(compare(prop, "ns_H", "ns_L", wn))
    rows.append(compare(prop, "wn_H", "wn_L", wn))
    rows.append(compare(prop, "ns_L", "wn_L", wn))
    rows.append(compare(prop, "ns_L", "wn_H", wn))
cmp = pd.DataFrame(rows)
def show(props):
    print(cmp[cmp.prop.isin(props)].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("Units are seconds for widths and latency (temporal), degrees for spatial width; gain in Hz per filtered unit.")
print("Spatial widths with failed DoG fits (NaN):", {c: int(inc[c + "_spatial_width"].isna().sum()) for c in ["ns_H", "ns_L"]},
      {c: int(wn[c + "_spatial_width"].isna().sum()) for c in ["wn_H", "wn_L"]})
show(["spatial_width"])

**Result: reproduced.** The spatial RF width does not change with contrast: 0.688 against 0.704 deg
for the movie (110 cells with both fits, p = 0.15) and 0.548 against 0.563 deg for the noise (31
cells, p = 0.46).

In [ ]:
show(["spatial_width"])
print("\nby subject, NS HC->LC:")
print(inc.groupby("subject")[["ns_H_spatial_width", "ns_L_spatial_width"]].mean().round(3).to_string())

**Result: reproduced.** From LC NS to LC WN the width falls from 0.694 to 0.563 deg, a decrease of
18.9% (P1 a) or 20.5% (P1 b), within 18 to 22%, paired t p = 3.9e-07 over 31 cells.

In [ ]:
show(["surround_ratio_a", "surround_ratio_b"])
print("\nby subject, NS:")
print(inc.groupby("subject")[["ns_H_surround_ratio_a", "ns_L_surround_ratio_a", "ns_H_surround_ratio_b", "ns_L_surround_ratio_b"]].mean().round(4).to_string())
print("centre sizes (pixels), median by condition:", {c: float(inc[c + "_n_center"].median()) for c in ["ns_H", "ns_L"]},
      {c: float(wn[c + "_n_center"].median()) for c in ["wn_H", "wn_L"]})

**Result: not reproduced.** For the movie, the surround/center ratio falls with contrast, by 11.6%
(P1 a) or 6.7% (P1 b) over 111 cells (p = 2.5e-05; 16.8% over the 31 Expt1 cells, p = 0.00073), below
the paper's 17%. For the noise it does not change (0.0595 against 0.0591, p = 0.93), against the
paper's 18%, so the white-noise test fails. Reading (b) of the surround peak gives the same.

In [ ]:
show(["surround_ratio_a", "surround_ratio_b"])

**Result: reproduced.** From LC NS to LC WN the surround/center ratio falls from 0.0964 to 0.0591,
a decrease of 38.7% (P1 a) or 37.2% (P1 b); 37.2% is within 30.6 to 37.4%. Paired t p = 2.7e-10 over
31 cells. With the opposite-sign surround peak (reading b) the decreases are 44.2% and 43.1%.

In [ ]:
show(["temporal_width"])
print("\nby subject, NS:")
print((inc.groupby("subject")[["ns_H_temporal_width", "ns_L_temporal_width"]].mean() * 1000).round(1).to_string())
print("temporal width range (ms):", {c: np.round(np.array([inc[c + '_temporal_width'].min(), inc[c + '_temporal_width'].max()]) * 1000, 1).tolist() for c in ["ns_H", "ns_L"]})

**Result: not reproduced.** For the movie the temporal width does not change with contrast over 111
cells (29.1 against 29.8 ms, +2.4%, p = 0.36); over the 31 Expt1 cells it widens by 16.6% (p = 0.042).
For the noise it widens by 43% (p = 0.0030), above the paper's 15% and not at p < 0.001. Widths are
read on a 16.7 ms lag grid, coarse for widths of 20 to 40 ms.

In [ ]:
show(["temporal_width"])

**Result: not reproduced.** From LC NS to LC WN the temporal width widens by 17.6% (P1 a) or 23.3%
(P1 b), against 15% (13.5 to 16.5%), and the paired t test gives p = 0.057.

## surround-temporal-correlation

> Across cells and the HC and LC natural movie conditions, the relative surround strength and the
> temporal width of the RF center are negatively correlated (r = -0.5, p < 0.001).

In [ ]:
for rd in ("a", "b"):
    x = np.r_[inc[f"ns_H_surround_ratio_{rd}"], inc[f"ns_L_surround_ratio_{rd}"]]
    y = np.r_[inc.ns_H_temporal_width, inc.ns_L_temporal_width]
    ok = np.isfinite(x) & np.isfinite(y)
    r, p = stats.pearsonr(x[ok], y[ok])
    print(f"reading {rd}: r = {r:.3f}, p = {p:.3g}, n = {ok.sum()} points ({len(inc)} cells x 2)")
    for sub in ("Expt1", "Expt2"):
        m = np.r_[inc.subject == sub, inc.subject == sub] & ok
        print(f"   {sub}: r = {stats.pearsonr(x[m], y[m])[0]:.3f}")

**Result: not reproduced.** Pooling HC and LC movie values of 111 cells, the correlation of
surround/center ratio with temporal width is r = -0.091 (p = 0.18) with reading (a) of the surround
peak and r = -0.145 (p = 0.031) with reading (b), against the paper's r = -0.5, p < 0.001.

## Nonlinearity properties (Figure 4)

> fig4b-gain-contrast: A decrease in contrast increases the gain of the nonlinearity by 89% for
> natural movies and 148% for white noise. fig4b-gain-correlations: a decrease in correlations
> increases the gain by 296%.

> fig4c-offset-contrast: A decrease in contrast decreases the (absolute) offset of the nonlinearity by
> 56% for natural movies and 59% for white noise. fig4c-offset-correlations-none: a decrease in
> correlations has no effect on the offset.

> fig4d-normalized-offset-contrast: A decrease in contrast increases the normalized offset by 14% for
> natural movies and 24% for white noise. fig4d-normalized-offset-correlations: a decrease in
> correlations increases it by 90%.

In [ ]:
# Offset adjusted per animal (P5): within each condition, each animal's mean q is subtracted and
# the grand mean added. Normalized offset is the adjusted q over the SD of the filtered stimulus.
def adjust(data, cols):
    out = data.copy()
    for c in cols:
        g = out.groupby("subject")[c + "_offset"].transform("mean")
        out[c + "_offset_adj"] = out[c + "_offset"] - g + out[c + "_offset"].mean()
        out[c + "_norm_offset_adj"] = out[c + "_offset_adj"] / out[c + "_ysd"]
    return out
inc = adjust(inc, ["ns_H", "ns_L"])
wn = wn.join(inc[["ns_H_offset_adj", "ns_L_offset_adj", "ns_H_norm_offset_adj", "ns_L_norm_offset_adj"]])
for c in ["wn_H", "wn_L"]:  # one animal: the adjustment adds nothing
    wn[c + "_offset_adj"] = wn[c + "_offset"]
    wn[c + "_norm_offset_adj"] = wn[c + "_offset"] / wn[c + "_ysd"]
rows = []
for prop in ["gain", "offset_adj", "offset", "norm_offset_adj", "norm_offset"]:
    rows.append(compare(prop, "ns_H", "ns_L", inc))
    rows.append(compare(prop, "ns_H", "ns_L", wn))
    rows.append(compare(prop, "wn_H", "wn_L", wn))
    rows.append(compare(prop, "ns_L", "wn_L", wn))
    rows.append(compare(prop, "ns_L", "wn_H", wn))
nlc = pd.DataFrame(rows)
cmp = pd.concat([cmp[~cmp.prop.isin(nlc.prop)], nlc], ignore_index=True)
print("NL fits that failed (NaN gain):", {c: int(inc[c + "_gain"].isna().sum()) for c in ["ns_H", "ns_L"]}, {c: int(wn[c + "_gain"].isna().sum()) for c in ["wn_H", "wn_L"]})
print("offsets q at or below the lowest NL point (no zero part):", {c: int((inc[c + "_offset"] <= inc[c + "_offset"].min()).sum()) for c in ["ns_H", "ns_L"]})
show(["gain"])
print("\nby subject, NS gain:")
print(inc.groupby("subject")[["ns_H_gain", "ns_L_gain"]].mean().round(5).to_string())

**Result: partially reproduced.** Gain rises at low contrast for the movie (p = 3.6e-42, 111 cells)
and the noise (p = 2.4e-10, 31 cells). The noise increase, 153.9% (P1 b), is within 133 to 163%; the
movie increase is 124.7% (P1 a) or 144% (P1 b), above the paper's 89% (80 to 98%).

In [ ]:
show(["gain"])

**Result: partially reproduced.** Gain rises from LC NS to LC WN (p = 3.5e-06), by 60.5% (P1 a) or
66.4% (P1 b), and the ratio of the means is 160.5%, all far below the paper's 296%.

In [ ]:
show(["offset_adj", "offset"])
print("\nby subject, NS offset (unadjusted):")
print(inc.groupby("subject")[["ns_H_offset", "ns_L_offset"]].mean().round(1).to_string())

**Result: reproduced.** The offset adjusted per animal falls at low contrast by 52.9% for the movie
(P1 a; 111 cells, p = 2.6e-13) and by 64.4% (P1 a) or 57.2% (P1 b) for the noise (31 cells,
p = 7.2e-10); 52.9% and 57.2% are within the paper's 56 and 59% (tolerance 5.6 and 5.9). Unadjusted,
the movie decrease is also 52.9% over 111 cells.

In [ ]:
show(["offset_adj"])

**Result: reproduced.** The adjusted offset does not differ between LC NS and LC WN: 1676 against
1360, -18.9%, paired t p = 0.16 over 31 cells. Without the adjustment p = 0.012 (a variant).

In [ ]:
show(["norm_offset_adj", "norm_offset"])
print("\nby subject, NS normalized offset (adjusted):")
print(inc.groupby("subject")[["ns_H_norm_offset_adj", "ns_L_norm_offset_adj"]].mean().round(3).to_string())

**Result: not reproduced.** The normalized offset rises at low contrast for the movie, by 42.3% (P1 a,
111 cells, p = 1.4e-05; the paper 14%), and does not change for the noise (0.676 against 0.676,
p = 0.99; the paper 24%).

In [ ]:
show(["norm_offset_adj"])

**Result: not reproduced.** From LC NS to LC WN the normalized offset rises by 50.1% (P1 a) or 26.3%
(P1 b), against 90%, with paired t p = 0.020, not below 0.001.

## figs1-property-correlations

> Changes in RF and NL properties are correlated with each other across stimulus conditions; every
> pairwise correlation of the properties that adapt significantly has p < 0.001, except
> surround/center ratio against temporal width for contrast during white noise (p = 0.06).

Readings: (a) the four properties that adapt in all three comparisons; (b) every property the paper
reports as changing significantly in that comparison (offset for the contrast comparisons, spatial
width for correlations).

In [ ]:
def pair_corrs(cols, conds, data):
    out = []
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            x = np.r_[data[conds[0] + "_" + a], data[conds[1] + "_" + a]].astype(float)
            y = np.r_[data[conds[0] + "_" + b], data[conds[1] + "_" + b]].astype(float)
            ok = np.isfinite(x) & np.isfinite(y)
            r, p = stats.pearsonr(x[ok], y[ok])
            out.append({"pair": f"{a} x {b}", "r": r, "p": p, "n": int(ok.sum())})
    return pd.DataFrame(out)
four = ["surround_ratio_a", "temporal_width", "gain", "norm_offset_adj"]
comps = {"contrast NS": (["ns_H", "ns_L"], inc, four + ["offset_adj"]),
         "contrast WN": (["wn_H", "wn_L"], wn, four + ["offset_adj"]),
         "correlations": (["ns_L", "wn_L"], wn, four + ["spatial_width"])}
s1 = []
for rd in ("a", "b"):
    for name, (conds, data, props) in comps.items():
        t = pair_corrs(four if rd == "a" else props, conds, data)
        t["comparison"], t["reading"] = name, rd
        s1.append(t)
s1 = pd.concat(s1, ignore_index=True)
s1["named"] = (s1.comparison == "contrast WN") & (s1.pair == "surround_ratio_a x temporal_width")
print(s1.round(4).to_string(index=False))
for rd in ("a", "b"):
    t = s1[s1.reading == rd]
    ok = ((t.p < 0.001) & ~t.named) | ((t.p > 0.001) & t.named)
    print(f"reading {rd}: {int(ok.sum())} of {len(t)} pairs as the legend states ({ok.mean():.0%}); named pair p = {t[t.named].p.iloc[0]:.3g}")

**Result: not reproduced.** Only 1 of 18 pairs (reading a) and 5 of 30 (reading b) behave as the
legend states; most pairs have p > 0.001. The named pair (surround/center ratio and temporal width,
contrast in WN) has p = 0.30.

## Reliability and sparseness (Figure 6)

> fig6a-reliability-contrast: A decrease in contrast decreases response reliability by 54% for
> natural movies and 74% for white noise (31 cells). fig6a-reliability-correlations: a decrease in
> correlations decreases it by 68%.

> fig6b-sparseness-contrast: A decrease in contrast causes only a small decrease in sparseness for
> natural movies (4%) and a larger one for white noise (16%). fig6b-sparseness-correlations: a
> decrease in correlations decreases sparseness by 14%.

> disc-lcns-hcwn-reliability-sparseness: Low-contrast natural movies and high-contrast white noise
> evoke responses of similar reliability and sparseness, those to white noise slightly higher.

> disc-lcns-hcwn-rf-nl-differences: The differences in spatial width, surround strength, gain and
> normalized offset between LC NS and LC WN remain significant when LC NS is compared with HC WN.

Reliability readings P8: (a) the signal power includes the mean, (b) it does not. `decrease_a_%`
and `decrease_b_%` are P1 (a) and (b).

In [ ]:
rs = []
for rd, pre in [("a", "rel_a_"), ("b", "rel_b_")]:
    for a, b in [("ns_H", "ns_L"), ("wn_H", "wn_L"), ("ns_L", "wn_L"), ("ns_L", "wn_H")]:
        x, y = wn[pre + a], wn[pre + b]
        ok = np.isfinite(x) & np.isfinite(y)
        x, y = x[ok], y[ok]
        t = stats.ttest_rel(x, y)
        pc = L.pct_change(x, y)
        rs.append({"reading": rd, "comparison": f"{a}->{b}", "n": int(ok.sum()), "mean_a": x.mean(), "mean_b": y.mean(), "sd_a": x.std(), "sd_b": y.std(),
                   "decrease_a_%": -pc["a"], "decrease_b_%": -pc["b"], "t": t.statistic, "p": t.pvalue})
rel = pd.DataFrame(rs)
print("cells with a non-finite reliability (no spikes in a block):", [f"{r.session} {r.unit_name}" for r in wn.itertuples()
      if not all(np.isfinite([getattr(r, "rel_a_" + c) for c in L.CONDITIONS]))])
print(f"{len(wn)} cells; non-finite reliabilities:", {c: int((~np.isfinite(wn["rel_a_" + c].astype(float))).sum()) for c in L.CONDITIONS})
print(rel.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nby penetration, reading a:")
print(wn.groupby("session")[["rel_a_ns_H", "rel_a_ns_L", "rel_a_wn_H", "rel_a_wn_L"]].mean().round(3).to_string())

**Result: partially reproduced.** Over 30 cells (one has a block without spikes), reliability falls
at low contrast for the movie (p = 3.1e-05) and the noise (p = 1.9e-09). The noise decrease, 70.2%
(P8 a) or 74.9% (P8 b) under P1 a, is within 66.6 to 81.4%. The movie decrease is 47.2% (P8 a) or
48.5% (P8 b), just below 48.6 to 59.4%.

In [ ]:
print(rel[rel.comparison == "ns_L->wn_L"].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** From LC NS to LC WN reliability falls by 62.7% (P8 a) or 67.7% (P8 b) under
P1 a, within 61.2 to 74.8%, paired t p = 1.7e-07 over 30 cells.

In [ ]:
sp = []
for a, b in [("ns_H", "ns_L"), ("wn_H", "wn_L"), ("ns_L", "wn_L"), ("ns_L", "wn_H")]:
    x, y = wn["sparse_" + a], wn["sparse_" + b]
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    t = stats.ttest_rel(x, y)
    pc = L.pct_change(x, y)
    sp.append({"comparison": f"{a}->{b}", "n": int(ok.sum()), "mean_a": x.mean(), "mean_b": y.mean(), "sd_a": x.std(), "sd_b": y.std(),
               "decrease_a_%": -pc["a"], "decrease_b_%": -pc["b"], "t": t.statistic, "p": t.pvalue})
spt = pd.DataFrame(sp)
print(spt.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nby penetration:")
print(wn.groupby("session")[["sparse_ns_H", "sparse_ns_L", "sparse_wn_H", "sparse_wn_L"]].mean().round(3).to_string())

**Result: partially reproduced.** Sparseness falls slightly for the movie (2.2%, p = 0.023) and more
for the noise (15.7%, p = 7.8e-12), so the noise decrease exceeds the movie decrease. The noise
decrease is within 14.4 to 17.6%; the movie decrease is below the paper's 4% (3.6 to 4.4%).

In [ ]:
print(spt[spt.comparison == "ns_L->wn_L"].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** From LC NS to LC WN sparseness falls by 13.1% (P1 a and b), within 12.6 to
15.4%; paired t p = 1.3e-08.

In [ ]:
print(rel[rel.comparison == "ns_L->wn_H"].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print(spt[spt.comparison == "ns_L->wn_H"].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** HC WN responses are more reliable (0.378 against 0.302, P8 a) and sparser
(0.856 against 0.831) than LC NS responses; neither is lower. Both differences are significant (p =
0.0058 and 0.00037), larger than the paper's "slightly higher".

In [ ]:
want = {"spatial_width": -1, "surround_ratio_a": -1, "gain": 1, "norm_offset_adj": 1}
t = cmp[(cmp.comparison == "ns_L->wn_H") & cmp.prop.isin(list(want) + ["surround_ratio_b"])].copy()
t["direction_ok"] = [np.sign(r.mean_b - r.mean_a) == want.get(r.prop, -1) for r in t.itertuples()]
print(t[["prop", "n", "mean_a", "mean_b", "pct_a", "t", "p", "direction_ok"]].round(4).to_string(index=False))
for lvl in (0.05, 0.001):
    m = t.prop.isin(want)
    print(f"level {lvl}: {int(((t.p < lvl) & t.direction_ok & m).sum())} of 4 significant in the direction of the LC NS -> LC WN change")

**Result: reproduced.** Comparing LC NS with HC WN over 31 cells, the spatial width (-21.1%,
p = 1.2e-15), the surround/center ratio (-38.3%, p = 6.3e-09) and the normalized offset (+50.2%,
p = 0.0048) differ in the direction of the LC NS to LC WN change: 3 of 4 at p < 0.05. The gain
differs in the opposite direction (-29.4%, p = 0.00024). At p < 0.001, 2 of 4.

## Generic models (Figures 7 and 8)

> fig7c-effective-contrast: Filtering by the RF raises the effective contrast of the natural movies
> relative to white noise, so that low-contrast natural movies and high-contrast white noise have
> similar effective contrasts despite different RMS contrasts.

> fig7d: Normalized to high-contrast natural movies, the effective contrasts of the four stimuli
> correspond closely to the observed reliability of LGN responses.

> fig8a: The natural movies filtered by the generic RFs have heavy-tailed distributions (kurtosis
> 4.06) while filtered white noise is Gaussian (kurtosis 3.01), and shuffling the movie frames
> before filtering removes the excess kurtosis.

> fig8-model-sparseness-shuffle: The generic LN model with the high-contrast natural movie RF and NL
> gives a response sparseness of 0.86 to the movie and 0.83 when the movie frames are shuffled.

> fig8b: The sparseness of the noiseless generic LN model increases as contrast and correlations
> decrease, the opposite of the experimental responses, matching the experiment only for
> high-contrast natural movies.

> fig8de: Adding Gaussian white noise to the filtered stimulus before the nonlinearity lowers model
> sparseness most for low-contrast white noise, and at one noise level the model and experimental
> sparseness agree closely for all four stimuli.

Generic RF: the mean of the cells' RFs (OFF sign-reversed), normalized to unit variance. Cells:
(a) the 31 Expt1 cells for all four stimuli, (b) all included cells with each stimulus. Stimulus
filtered: (a) the long segments, (b) the 12 s repeat clips, at 16 window positions on a 9-pixel
grid of the 48 x 48 frame (a standard choice). Effective contrast is the SD of the filtered
stimulus; RMS contrast the SD of the stimulus, both in gray levels. The generic NL uses the mean
gain and the offset under readings (a) mean normalized offset x SD of the filtered stimulus, (b)
mean adjusted offset.

In [ ]:
def generic_rf(data, cond):
    rfs = [units[(r.subject, r.session, r.unit_name)][cond]["rf"] * (1 if r.onoff == "ON" else -1) for r in data.itertuples()]
    g = np.mean(rfs, 0)
    return g / g.std()
positions = [(r0, c0) for r0 in (0, 9, 18, 27) for c0 in (0, 9, 18, 27)]
def stim_long(cond):
    tm = "NAT" if cond.startswith("ns") else "WN"
    n = 54000 if tm == "NAT" else int(round(L.WN_SECONDS * L.FRAME_RATE))
    return T[tm][:n], {"ns_H": 0.40, "ns_L": 0.15, "wn_H": 0.55, "wn_L": 0.20}[cond]
def stim_clips(cond):
    k = "clip_nr_b" if cond.startswith("ns") else "clip_wr_b"
    return [clips[k + str(i)] for i in (1, 2, 3)], {"ns_H": 0.40, "ns_L": 0.15, "wn_H": 0.55, "wn_L": 0.20}[cond]
def filt(F, rms, g):
    return np.concatenate([L.filtered(np.ascontiguousarray(F[:, r0:r0 + L.WIN, c0:c0 + L.WIN]).reshape(len(F), -1) * np.float32(125 * rms), g)[L.NLAGS - 1:]
                           for r0, c0 in positions])
def filtered_stim(cond, g, which):
    if which == "a":
        F, rms = stim_long(cond)
        return filt(F, rms, g), 125 * rms * float(F.std())
    Fs, rms = stim_clips(cond)
    return np.concatenate([filt(F, rms, g) for F in Fs]), 125 * rms * float(np.concatenate(Fs).std())
groups = {"a": {c: wn for c in L.CONDITIONS}, "b": {"ns_H": inc, "ns_L": inc, "wn_H": wn, "wn_L": wn}}
G, Y, eff = {}, {}, []
for gr in ("a", "b"):
    for c in L.CONDITIONS:
        G[gr, c] = generic_rf(groups[gr][c], c)
        for st in ("a", "b"):
            y, rmsg = filtered_stim(c, G[gr, c], st)
            Y[gr, st, c] = y
            eff.append({"cells": gr, "stimulus": st, "cond": c, "n_cells": len(groups[gr][c]), "rms_contrast_gray": rmsg,
                        "effective": float(y.std()), "ratio": float(y.std()) / rmsg})
eff = pd.DataFrame(eff)
print(eff.round(3).to_string(index=False))
for (gr, st), t in eff.groupby(["cells", "stimulus"]):
    t = t.set_index("cond")
    ratio_ok = min(t.ratio["ns_H"], t.ratio["ns_L"]) > max(t.ratio["wn_H"], t.ratio["wn_L"])
    d = {c: abs(t.effective["ns_L"] - t.effective[c]) for c in ["ns_H", "wn_H", "wn_L"]}
    print(f"cells {gr}, stimulus {st}: NS ratios above WN ratios: {ratio_ok}; |LC NS - x| effective: "
          f"{ {k: round(v, 1) for k, v in d.items()} }; closest to HC WN: {min(d, key=d.get) == 'wn_H'}")

**Result: reproduced.** Under every reading, the ratio of effective to RMS contrast is 243 to 277 for
the movies and 72 to 73 for the noise, and the LC NS effective contrast (4763 to 5185) is closest to
that of HC WN (4997 to 5010).

In [ ]:
fd = []
for gr in ("a", "b"):
    for st in ("a", "b"):
        e = eff[(eff.cells == gr) & (eff.stimulus == st)].set_index("cond").effective
        for rd in ("a", "b"):
            m = wn[[f"rel_{rd}_{c}" for c in L.CONDITIONS]].mean().values
            s = wn[[f"rel_{rd}_{c}" for c in L.CONDITIONS]].std().values
            en = e[L.CONDITIONS].values / e["ns_H"]
            mn, sn = m / m[0], s / m[0]
            same_rank = list(np.argsort(en)) == list(np.argsort(mn))
            within = np.abs(en - mn) <= sn
            fd.append({"cells": gr, "stimulus": st, "reliability": rd, "eff_norm": np.round(en, 3).tolist(), "rel_norm": np.round(mn, 3).tolist(),
                       "rel_sd_norm": np.round(sn, 3).tolist(), "same_rank": same_rank, "n_within_sd": int(within.sum())})
print(pd.DataFrame(fd).to_string(index=False))

**Result: reproduced, under the 12 s clip reading of the stimulus.** Normalized to HC NS, the
effective contrasts are 1, 0.384, 0.404 and 0.146 (HC NS, LC NS, HC WN, LC WN) and the reliabilities
1, 0.528, 0.662 and 0.197: same ranks, each within one normalized SD. With the long segments the
effective contrasts of LC NS and HC WN swap order (0.390 and 0.382), so the ranks differ.

In [ ]:
rng = np.random.default_rng(L.seed("fig8a-filtered-kurtosis"))
kt = []
for gr in ("a", "b"):
    for st in ("a", "b"):
        row = {"cells": gr, "stimulus": st}
        for c in L.CONDITIONS:
            row[c] = stats.kurtosis(Y[gr, st, c], fisher=False)
        kt.append(row)
kt = pd.DataFrame(kt)
print("kurtosis (Gaussian = 3) of the filtered stimuli:")
print(kt.round(3).to_string(index=False))
F, rms = stim_long("ns_H")
sh = []
for k in range(5):
    rng_k = np.random.default_rng(L.seed("fig8a-filtered-kurtosis") + 1000 * k)
    perm = rng_k.permutation(len(F))
    sh.append([stats.kurtosis(filt(F[perm], rms, G[gr, "ns_H"]), fisher=False) for gr in ("a", "b")])
sh = np.array(sh)
print("frame-shuffled HC NS filtered by the HC NS generic RF, kurtosis (cells a, b), claim seed:", np.round(sh[0], 3),
      "; over the five seeds:", np.round(sh.min(0), 3), "to", np.round(sh.max(0), 3))

**Result: reproduced.** Filtered with the generic RFs over the long segments, the movies have
kurtosis 4.01 (HC) and 3.79 (LC), within 3.65 to 4.47, and the noise 2.96 and 2.98, within 2.71 to
3.31. Frame-shuffled HC NS has kurtosis 3.18 (3.16 to 3.19 over five seeds), within 0.30 of 3. Over
the 12 s clips the movie values are lower (3.44 to 3.55).

In [ ]:
def model_sparse(y, gain, q):
    return L.sparseness(gain * np.maximum(y - q, 0))
gm = []
for gr in ("a", "b"):
    for st in ("a", "b"):
        for rd in ("a", "b"):
            row = {"cells": gr, "stimulus": st, "offset": rd}
            for c in L.CONDITIONS:
                data = groups[gr][c]
                y = Y[gr, st, c]
                q = data[c + "_norm_offset_adj"].mean() * y.std() if rd == "a" else data[c + "_offset_adj"].mean()
                row[c] = model_sparse(y, data[c + "_gain"].mean(), q)
            gm.append(row)
gm = pd.DataFrame(gm)
print("model sparseness:")
print(gm.round(3).to_string(index=False))
F, rms = stim_long("ns_H")
for gr in ("a", "b"):
    data = groups[gr]["ns_H"]
    vals = []
    for k in range(5):
        perm = np.random.default_rng(L.seed("fig8-model-sparseness-shuffle") + 1000 * k).permutation(len(F))
        y = filt(F[perm], rms, G[gr, "ns_H"])
        vals.append([model_sparse(y, data.ns_H_gain.mean(), data.ns_H_norm_offset_adj.mean() * y.std()),
                     model_sparse(y, data.ns_H_gain.mean(), data.ns_H_offset_adj.mean())])
    vals = np.array(vals)
    print(f"cells {gr}: frame-shuffled HC NS model sparseness, offset a {vals[0, 0]:.3f}, offset b {vals[0, 1]:.3f} (claim seed); "
          f"over five seeds {np.round(vals.min(0), 3)} to {np.round(vals.max(0), 3)}")

**Result: reproduced.** The generic model with the HC NS RF and NL gives sparseness 0.810 for the
movie and 0.778 for the frame-shuffled movie (offset reading a, 31 cells, long segments), within the
tolerances of 0.86 and 0.83, and the shuffled value is lower. The shuffled value is 0.778 to 0.779
over five seeds.

In [ ]:
exp_m = wn[[f"sparse_{c}" for c in L.CONDITIONS]].mean().values
exp_s = wn[[f"sparse_{c}" for c in L.CONDITIONS]].std().values
print("experimental sparseness (12 s repeats):", dict(zip(L.CONDITIONS, np.round(exp_m, 3))), "SD", dict(zip(L.CONDITIONS, np.round(exp_s, 3))))
for r in gm.itertuples():
    mv = np.array([getattr(r, c) for c in L.CONDITIONS])
    print(f"cells {r.cells} stim {r.stimulus} offset {r.offset}: lowest {L.CONDITIONS[int(np.argmin(mv))]}, highest {L.CONDITIONS[int(np.argmax(mv))]}, "
          f"HC NS model {mv[0]:.3f} within 1 SD of experiment: {abs(mv[0] - exp_m[0]) <= exp_s[0]}")
print("experiment HC NS above LC WN:", exp_m[0] > exp_m[3])

**Result: reproduced.** With the 31 cells, the long segments and offset reading (a), the noiseless
model is least sparse for HC NS (0.810) and most for LC WN (0.851); the experiment is sparser for HC
NS (0.850) than LC WN (0.722); and the model HC NS value is within one SD (0.049) of the experiment.
Under the 12 s clips the model HC NS value (0.796) is outside one SD.

In [ ]:
# Noise added to the filtered stimulus before the NL (Figure 8C), on the 12 s repeat stimuli
# (stimulus reading b), cells reading a, offset reading a. Noise SD in filtered-stimulus units.
levels = np.linspace(0, 3000, 31)
def noisy_sparse(c, sd_noise, rng, trials):
    data = groups["a"][c]
    y = Y["a", "b", c]
    q = data[c + "_norm_offset_adj"].mean() * y.std()
    if trials == 1:
        return model_sparse(y + rng.standard_normal(len(y)) * sd_noise, data[c + "_gain"].mean(), q)
    acc = np.zeros(len(y))
    for _ in range(trials):
        acc += data[c + "_gain"].mean() * np.maximum(y + rng.standard_normal(len(y)) * sd_noise - q, 0)
    return L.sparseness(acc / trials)
out8 = {}
for rd, trials in (("a", 1), ("b", 120)):
    rng = np.random.default_rng(L.seed("fig8de-noise-reconciles") + (0 if rd == "a" else 1))
    out8[rd] = np.array([[noisy_sparse(c, s, rng, trials) for c in L.CONDITIONS] for s in levels])
for k in (1000, 2000, 3000, 4000):
    for rd, trials in (("a", 1), ("b", 120)):
        rng = np.random.default_rng(L.seed("fig8de-noise-reconciles") + (0 if rd == "a" else 1) + k)
        M = np.array([[noisy_sparse(c, s, rng, trials) for c in L.CONDITIONS] for s in levels])
        print(f"seed + {k}, reading {rd}: largest decrease for {L.CONDITIONS[int(np.argmax(M[0] - M[-1]))]}; "
              f"levels with all four within 1 SD: {int((np.abs(M - exp_m) <= exp_s).all(1).sum())}; HC NS model max {M[:, 0].max():.3f}")
for rd, M in out8.items():
    drop = M[0] - M.min(0)
    within = (np.abs(M - exp_m) <= exp_s).all(1)
    print(f"reading {rd}: decrease from no noise to the largest level: {dict(zip(L.CONDITIONS, np.round(M[0] - M[-1], 3)))}; "
          f"largest for {L.CONDITIONS[int(np.argmax(M[0] - M[-1]))]}; levels with all four within 1 SD: {levels[within].tolist()}")
    print(pd.DataFrame(M[::5], index=levels[::5], columns=L.CONDITIONS).round(3).to_string())

**Result: not reproduced.** Added noise lowers model sparseness most for LC WN under both readings,
but at no noise level are all four model values within one SD of the experiment: the HC NS model
value is at most 0.797, below 0.850 - 0.049, under every level and seed.

## Claims from the Experimental Procedures

> methods-latency-biphasic: Changes in contrast and correlations significantly change the latency
> (time to peak) of the RF center's temporal profile but not its biphasic ratio.

> methods-offset-varies-across-animals: Of the RF and NL parameters, only the offset q differs
> significantly across animals within a stimulus condition.

Latency is read on the frame grid (16.7 ms), so it changes in steps of one frame.

In [ ]:
lt = cmp[cmp.prop.isin(["latency", "biphasic"]) & cmp.comparison.isin(["ns_H->ns_L", "wn_H->wn_L", "ns_L->wn_L"])]
lt = lt[~((lt.comparison == "ns_H->ns_L") & (lt.group != f"all {len(inc)}"))]
print(lt[["prop", "comparison", "n", "mean_a", "mean_b", "t", "p"]].round(4).to_string(index=False))
for lvl in (0.05, 0.001):
    print(f"latency significant at {lvl}: {int((lt[lt.prop == 'latency'].p < lvl).sum())} of 3; biphasic significant at 0.05: {int((lt[lt.prop == 'biphasic'].p < 0.05).sum())} of 3")

**Result: partially reproduced.** Latency changes with contrast for the noise (p = 0.0006) and with
correlations (p = 0.013), but not with contrast for the movie (p = 0.14): 2 of 3 at p < 0.05. The
biphasic ratio changes in none (p = 0.34 to 0.90). Latency is read on the 16.7 ms frame grid.

In [ ]:
an = []
for c in ["ns_H", "ns_L"]:
    for prop in ["spatial_width", "surround_ratio_a", "temporal_width", "gain", "offset", "norm_offset"]:
        g1, g2 = [inc[inc.subject == s][c + "_" + prop].dropna() for s in ("Expt1", "Expt2")]
        an.append({"cond": c, "prop": prop, "mean_Expt1": g1.mean(), "mean_Expt2": g2.mean(), "anova_p": stats.f_oneway(g1, g2).pvalue,
                   "kruskal_p": stats.kruskal(g1, g2).pvalue})
an = pd.DataFrame(an)
print(an.round(4).to_string(index=False))
for lvl in (0.05, 0.001):
    for t in ("anova_p", "kruskal_p"):
        sig = an[an[t] < lvl]
        print(f"{t} < {lvl}: offset in {int((sig.prop == 'offset').sum())} conditions; other parameters significant: {sorted(set(sig.prop) - {'offset'})}")
print("White noise was recorded in one animal (Expt1) only, so no test across animals is possible for it.")

**Result: not reproduced.** Between Expt1 and Expt2 the offset differs at low contrast (ANOVA
p = 0.021) but not at high contrast (p = 0.46), and spatial width, surround/center ratio, temporal
width and gain also differ (p from 0.0001 to 0.047). Only two animals can be compared, and only for
the movie.

## Cell types

> celltype-y-wider-than-x: The spatial RF center is wider for Y than X cells during high-contrast
> (0.75 against 0.63 deg) and low-contrast (0.73 against 0.63 deg) natural movies.

> celltype-y-stronger-surround: The relative surround strength is larger for Y than X cells during
> high-contrast natural movies (0.16 against 0.13).

> celltype-off-wider-temporal: During low-contrast white noise the temporal profile of the RF
> center is wider for OFF than ON cells (29 against 25 ms).

> celltype-prediction-x-y: LN model predictions are slightly more accurate for X than Y cells at
> high (0.73 against 0.69) and low (0.78 against 0.75) contrast, but the differences are not
> significant.

In [ ]:
xy = inc[inc.xy.isin(["X", "Y"])]
print("X and Y cells:", xy.xy.value_counts().to_dict(), "; by subject:", xy.groupby(["subject", "xy"]).size().to_dict())
for c in ["ns_H", "ns_L"]:
    a, b = xy[xy.xy == "X"][c + "_spatial_width"].dropna(), xy[xy.xy == "Y"][c + "_spatial_width"].dropna()
    t = stats.ttest_ind(a, b)
    print(f"{c}: X {a.mean():.3f} ({a.std():.3f}, n={len(a)}), Y {b.mean():.3f} ({b.std():.3f}, n={len(b)}) deg; t = {t.statistic:.2f}, p = {t.pvalue:.3g}")

**Result: reproduced.** The spatial width is larger for Y than X cells at high contrast (0.743
against 0.646 deg, p = 0.00086) and low contrast (0.778 against 0.642 deg, p = 0.00013), all four
means within the paper's SDs.

In [ ]:
for rd in ("a", "b"):
    a, b = xy[xy.xy == "X"][f"ns_H_surround_ratio_{rd}"].dropna(), xy[xy.xy == "Y"][f"ns_H_surround_ratio_{rd}"].dropna()
    t = stats.ttest_ind(a, b)
    print(f"reading {rd}: X {a.mean():.3f} ({a.std():.3f}, n={len(a)}), Y {b.mean():.3f} ({b.std():.3f}, n={len(b)}); t = {t.statistic:.2f}, p = {t.pvalue:.3g}")

**Result: not reproduced.** At high contrast the surround/center ratio is 0.105 for X and 0.113 for Y
cells, both within the paper's SDs, but the difference is not significant (p = 0.27).

In [ ]:
a, b = wn[wn.onoff == "ON"].wn_L_temporal_width * 1000, wn[wn.onoff == "OFF"].wn_L_temporal_width * 1000
t = stats.ttest_ind(a, b)
print(f"LC WN temporal width: ON {a.mean():.1f} (SD {a.std():.1f}, SE {a.sem():.1f}, n={len(a)}), OFF {b.mean():.1f} (SD {b.std():.1f}, SE {b.sem():.1f}, n={len(b)}) ms; "
      f"t = {t.statistic:.2f}, p = {t.pvalue:.3g}")

**Result: not reproduced.** At LC WN the temporal width is 37.0 ms for ON and 35.9 ms for OFF cells
(p = 0.84), against 25 and 29 ms.

In [ ]:
pxy = pr[pr.xy.isin(["X", "Y"])]
print("classified cells of the prediction set:", pxy.xy.value_counts().to_dict())
for rd in ("a", "b"):
    for c in ("H", "L"):
        a, b = pxy[pxy.xy == "X"][f"{rd}_{c}"], pxy[pxy.xy == "Y"][f"{rd}_{c}"]
        t = stats.ttest_ind(a, b)
        print(f"reading {rd}, {c}C: X {a.mean():.3f} (n={len(a)}), Y {b.mean():.3f} (n={len(b)}); t = {t.statistic:.2f}, p = {t.pvalue:.3g}")

**Result: not reproduced.** The X and Y predictions do not differ (p = 0.79 and 0.51, reading a), as
the paper says, but the means (0.471 and 0.482 at HC, 0.516 and 0.548 at LC) are far below 0.73, 0.69,
0.78 and 0.75, and Y is not below X.

## Summary

| Claim | Outcome |
|---|---|
| sample-composition | not reproduced |
| firing-rate-contrast | partially reproduced |
| firing-rate-lc-ns-wn-similar | reproduced (assumption) |
| ln-prediction-accuracy | not reproduced |
| fig3b-spatial-width-contrast | reproduced |
| fig3b-spatial-width-correlations | reproduced |
| fig3c-surround-contrast | not reproduced |
| fig3c-surround-correlations | reproduced |
| fig3d-temporal-width-contrast | not reproduced |
| fig3d-temporal-width-correlations | not reproduced |
| surround-temporal-correlation | not reproduced |
| figs1-property-correlations | not reproduced |
| fig4b-gain-contrast | partially reproduced |
| fig4b-gain-correlations | partially reproduced |
| fig4c-offset-contrast | reproduced |
| fig4c-offset-correlations-none | reproduced |
| fig4d-normalized-offset-contrast | not reproduced |
| fig4d-normalized-offset-correlations | not reproduced |
| fig5-within-trial-contrast | blocked (not run within the author's time) |
| fig6a-reliability-contrast | partially reproduced |
| fig6a-reliability-correlations | reproduced |
| fig6b-sparseness-contrast | partially reproduced |
| fig6b-sparseness-correlations | reproduced |
| fig7c-effective-contrast | reproduced |
| fig7d-reliability-tracks-effective-contrast | reproduced |
| fig8a-filtered-kurtosis | reproduced |
| fig8-model-sparseness-shuffle | reproduced |
| fig8b-model-vs-experiment-sparseness | reproduced |
| fig8de-noise-reconciles | not reproduced |
| disc-lcns-hcwn-reliability-sparseness | reproduced |
| disc-lcns-hcwn-rf-nl-differences | reproduced |
| methods-latency-biphasic | partially reproduced |
| methods-offset-varies-across-animals | not reproduced |
| celltype-y-wider-than-x | reproduced |
| celltype-y-stronger-surround | not reproduced |
| celltype-off-wider-temporal | not reproduced |
| celltype-prediction-x-y | not reproduced |
| methods-spherical-asymmetry | blocked (not run within the author's time) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** RFs use 12 lags of one stimulus frame (16.7 ms) in
place of 24 lags of 8 ms, so temporal widths and latencies are coarse. The ridge penalty is chosen
by held-out prediction, as the paper chose D, but its scale and its being relative to the stimulus
variance are assumptions. The paper's 68-cell sample cannot be identified: the inclusion rule passes
111 cells here, and every comparison uses those. The 90 s test segment of Expt2 is part of the
15 min movie, so its frames were left out of the training data. The first 6 min of the longer noise
recordings are used. ON or OFF is taken from the sign of the RF. Figure 5 and the simulation of the
Experimental Procedures were not run.

**What was found about the dataset.** The data the paper needs are there for two animals (Expt1,
Expt2), with the stimulus frames, and the clocks agree (the 90 s repeats correlate with the same
frames of the unique movie at zero shift). The 90 s cross-validation segment is the first 90 s of the
15 min movie, not a different segment as the paper says; the noise recordings last 10 and 15 min, not
6; the units carry no ON/OFF label; and nothing marks which units or penetrations the paper used.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")